# Titanic Dataset — Feature Engineering & Analysis
**Assignment 2 | Artificial Intelligence**

---
This notebook covers:
- **Part 1**: Data Cleaning (missing values, outliers, consistency)
- **Part 2**: Feature Engineering (derived, encoded, transformed features)
- **Part 3**: Feature Selection (correlation analysis, RF importance, RFE)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid', palette='muted')

# Make sure scripts directory is importable
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))


---
## Part 1: Data Cleaning
### 1.1 Load and Inspect

In [ ]:
df_raw = pd.read_csv('../data/train.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe()

### 1.2 Missing Value Analysis

In [ ]:
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Pct (%)': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False)

fig, ax = plt.subplots()
bars = ax.bar(missing_df.index, missing_df['Pct (%)'], color=['#e74c3c','#f39c12','#3498db'])
ax.set_title('Missing Value Percentage by Column', fontsize=14, fontweight='bold')
ax.set_ylabel('% Missing')
for bar, pct in zip(bars, missing_df['Pct (%)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pct:.1f}%', ha='center', fontsize=11)
plt.tight_layout()
plt.show()
print(missing_df)

#### Decision rationale:
| Column | Strategy | Reason |
|--------|----------|---------|
| **Age** (~20% missing) | Median imputation + indicator flag | Median is robust to outliers; flag preserves missingness signal |
| **Cabin** (~77% missing) | Extract deck letter; label unknown 'U' | Too many missing to impute reliably; deck captures class signal |
| **Embarked** (2 missing) | Mode imputation | Negligible count; mode is straightforward |


In [ ]:
df = df_raw.copy()

# Age
age_median = df['Age'].median()
df['Age_Missing'] = df['Age'].isnull().astype(int)
df['Age'].fillna(age_median, inplace=True)

# Embarked
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# Cabin → Deck
df['Deck'] = df['Cabin'].apply(lambda x: str(x)[0] if pd.notna(x) else 'U')

print('Missing after cleaning:')
print(df.isnull().sum()[df.isnull().sum() > 0])

### 1.3 Outlier Detection and Handling

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ['Fare', 'Age']):
    ax.boxplot(df[col], patch_artist=True,
               boxprops=dict(facecolor='#3498db', color='navy'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(f'{col} — Before Capping', fontweight='bold')
    ax.set_ylabel(col)
plt.tight_layout()
plt.show()

In [ ]:
# Cap at 1st and 99th percentile (Winsorisation)
for col in ['Fare', 'Age']:
    q1, q99 = df[col].quantile([0.01, 0.99])
    n_out = ((df[col] < q1) | (df[col] > q99)).sum()
    df[col] = df[col].clip(lower=q1, upper=q99)
    print(f'{col}: capped {n_out} outliers → range [{q1:.2f}, {q99:.2f}]')

### 1.4 Data Consistency

In [ ]:
# Standardise text columns
df['Sex'] = df['Sex'].str.lower().str.strip()
df['Embarked'] = df['Embarked'].str.upper().str.strip()

# Duplicates
n_dupes = df.duplicated().sum()
df.drop_duplicates(inplace=True)
print(f'Duplicates removed: {n_dupes}')
print(f'Sex unique values: {df["Sex"].unique()}')
print(f'Embarked unique values: {df["Embarked"].unique()}')

In [ ]:
# Save cleaned dataset
df.to_csv('../data/train_cleaned.csv', index=False)
print(f'Cleaned dataset saved → data/train_cleaned.csv  shape={df.shape}')

---
## Part 2: Feature Engineering
### 2.1 Derived Features

In [ ]:
# FamilySize and IsAlone
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone']    = (df['FamilySize'] == 1).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
survival_by_family = df.groupby('FamilySize')['Survived'].mean().reset_index()
axes[0].bar(survival_by_family['FamilySize'], survival_by_family['Survived'],
            color='#2ecc71', edgecolor='darkgreen')
axes[0].set_title('Survival Rate by Family Size', fontweight='bold')
axes[0].set_xlabel('FamilySize')
axes[0].set_ylabel('Survival Rate')

alone_surv = df.groupby('IsAlone')['Survived'].mean()
axes[1].bar(['Not Alone', 'Alone'], alone_surv.values, color=['#3498db','#e74c3c'])
axes[1].set_title('Survival Rate: Alone vs Not Alone', fontweight='bold')
axes[1].set_ylabel('Survival Rate')
plt.tight_layout()
plt.show()

In [ ]:
# Title extraction
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')
df['Title'] = df['Title'].str.strip()

rare = {'Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'}
df['Title'] = df['Title'].apply(lambda t: 'Rare' if t in rare else t)
df['Title'] = df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})

title_surv = df.groupby('Title')['Survived'].agg(['mean','count']).reset_index()
title_surv.columns = ['Title','SurvivalRate','Count']
title_surv = title_surv.sort_values('SurvivalRate', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(title_surv)))
bars = ax.bar(title_surv['Title'], title_surv['SurvivalRate'], color=colors)
ax.set_title('Survival Rate by Extracted Title', fontweight='bold')
ax.set_ylabel('Survival Rate')
for bar, count in zip(bars, title_surv['Count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'n={count}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Age groups
df['AgeGroup'] = pd.cut(df['Age'], bins=[0,12,18,60,200],
                         labels=['Child','Teen','Adult','Senior'])

# Fare per person
df['FarePerPerson'] = df['Fare'] / df['FamilySize']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

age_surv = df.groupby('AgeGroup', observed=True)['Survived'].mean()
axes[0].bar(age_surv.index, age_surv.values, color=['#1abc9c','#3498db','#9b59b6','#e67e22'])
axes[0].set_title('Survival Rate by Age Group', fontweight='bold')
axes[0].set_ylabel('Survival Rate')

axes[1].hist(df['FarePerPerson'], bins=40, color='#3498db', edgecolor='white')
axes[1].set_title('Distribution of Fare Per Person', fontweight='bold')
axes[1].set_xlabel('Fare Per Person')
plt.tight_layout()
plt.show()

### 2.2 Feature Transformations (Log Scaling)

In [ ]:
df['Fare_Log']          = np.log1p(df['Fare'])
df['FarePerPerson_Log'] = np.log1p(df['FarePerPerson'])
df['Age_Log']           = np.log1p(df['Age'])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].hist(df['Fare'], bins=50, color='#e74c3c', edgecolor='white')
axes[0,0].set_title('Fare — Original (Skewed)', fontweight='bold')

axes[0,1].hist(df['Fare_Log'], bins=50, color='#2ecc71', edgecolor='white')
axes[0,1].set_title('Fare — Log Transformed', fontweight='bold')

axes[1,0].hist(df['Age'], bins=40, color='#e74c3c', edgecolor='white')
axes[1,0].set_title('Age — Original', fontweight='bold')

axes[1,1].hist(df['Age_Log'], bins=40, color='#2ecc71', edgecolor='white')
axes[1,1].set_title('Age — Log Transformed', fontweight='bold')

plt.suptitle('Log Transformation Effect on Skewed Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.3 Categorical Encoding (One-Hot)

In [ ]:
df_enc = df.copy()

# Drop raw columns not needed for modelling
drop_cols = ['PassengerId','Name','Ticket','Cabin','Fare','Age','FarePerPerson']
df_enc.drop(columns=[c for c in drop_cols if c in df_enc.columns], inplace=True)

# One-hot encode
ohe_cols = ['Sex','Embarked','Title','Deck','AgeGroup','FamilyType'] if 'FamilyType' in df_enc.columns else ['Sex','Embarked','Title','Deck','AgeGroup']
df_enc = pd.get_dummies(df_enc, columns=[c for c in ohe_cols if c in df_enc.columns], drop_first=False)

bool_cols = df_enc.select_dtypes('bool').columns
df_enc[bool_cols] = df_enc[bool_cols].astype(int)

print(f'Shape after encoding: {df_enc.shape}')
df_enc.head()

### 2.4 Interaction Features

In [ ]:
df_enc['Pclass_Fare'] = df_enc['Pclass'] * df_enc['Fare_Log']
df_enc['Pclass_Age']  = df_enc['Pclass'] * df_enc['Age_Log']

# Visualise: Pclass × Fare interaction vs Survival
fig, ax = plt.subplots(figsize=(10, 4))
for p, col in zip([1,2,3], ['#f1c40f','#3498db','#e74c3c']):
    mask = df['Pclass'] == p
    ax.hist(df.loc[mask, 'Fare_Log'], bins=30, alpha=0.6, label=f'Pclass {p}', color=col)
ax.set_title('Log(Fare) Distribution by Passenger Class', fontweight='bold')
ax.set_xlabel('Fare_Log')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save feature-engineered dataset
df_enc.to_csv('../data/train_features.csv', index=False)
print(f'Feature-engineered dataset saved → data/train_features.csv  shape={df_enc.shape}')

---
## Part 3: Feature Selection
### 3.1 Correlation Analysis

In [ ]:
df_feat = pd.read_csv('../data/train_features.csv')

corr_matrix = df_feat.corr().abs()

# Plot heatmap of top 20 features by correlation with Survived
top20_cols = corr_matrix['Survived'].sort_values(ascending=False).head(21).index
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(df_feat[top20_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix — Top 20 Features', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Remove pairs with correlation > 0.90
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
drop_corr = [col for col in upper.columns if any(upper[col] > 0.90)]
print(f'Dropping highly correlated features (>0.90): {drop_corr}')
df_feat.drop(columns=drop_corr, inplace=True)
print(f'Shape after correlation filter: {df_feat.shape}')

### 3.2 Random Forest Feature Importance

In [ ]:
X = df_feat.drop(columns=['Survived'])
y = df_feat['Survived']

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X, y)

importance = pd.Series(rf.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

top20 = importance.head(20)
fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.viridis(np.linspace(0.2, 0.85, len(top20)))
ax.barh(top20.index[::-1], top20.values[::-1], color=colors[::-1])
ax.set_title('Random Forest Feature Importances (Top 20)', fontweight='bold', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

top20_features = top20.index.tolist()
print('Top 20 features:', top20_features)

### 3.3 Recursive Feature Elimination (RFE) — Bonus

In [ ]:
X_top = df_feat[top20_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_top)

rf_rfe = RandomForestClassifier(n_estimators=100, random_state=42)
rfe = RFE(estimator=rf_rfe, n_features_to_select=15, step=1)
rfe.fit(X_scaled, y)

selected = X_top.columns[rfe.support_].tolist()
dropped  = X_top.columns[~rfe.support_].tolist()
print('RFE selected:', selected)
print('RFE dropped:', dropped)

In [ ]:
# Visualise selected vs dropped
rfe_df = pd.DataFrame({'Feature': X_top.columns,
                        'Selected': rfe.support_,
                        'Rank': rfe.ranking_})
rfe_df = rfe_df.sort_values('Rank')

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2ecc71' if s else '#e74c3c' for s in rfe_df['Selected']]
ax.barh(rfe_df['Feature'][::-1], rfe_df['Rank'][::-1], color=colors[::-1])
ax.set_title('RFE Feature Ranking (Green = Selected, Red = Dropped)', fontweight='bold')
ax.set_xlabel('RFE Rank (1 = best)')
plt.tight_layout()
plt.show()

In [ ]:
# Final selected dataset
df_final = df_feat[selected + ['Survived']]
df_final.to_csv('../data/train_selected.csv', index=False)
print(f'Final dataset saved → data/train_selected.csv  shape={df_final.shape}')
print('\nSelected features:')
for f in selected:
    print(f'  {f}')

---
## Summary

### Data Cleaning Decisions
| Column | Strategy | Rationale |
|--------|----------|-----------|
| Age (~20% missing) | Median imputation + binary indicator | Median avoids skew; indicator preserves missingness information |
| Cabin (~77% missing) | Deck letter extraction; label unknown 'U' | Too sparse to impute; deck letter still carries class signal |
| Embarked (2 missing) | Mode imputation | Negligible; mode is safe |
| Fare/Age outliers | 1st–99th percentile capping | Removes extreme values without data loss |

### Features Engineered
- **FamilySize**, **IsAlone** — group/social context
- **Title** — gender + social status proxy (Mr, Mrs, Miss, Rare)
- **Deck** — cabin/class proxy
- **AgeGroup** — non-linear age effect (Child, Teen, Adult, Senior)
- **FarePerPerson** — economic signal per traveller
- **Fare_Log, Age_Log, FarePerPerson_Log** — normalise skew
- **Pclass_Fare, Pclass_Age** — interaction features
- **One-hot encoding** — Sex, Embarked, Title, Deck, AgeGroup

### Key Findings
- **Sex and Title** are the strongest predictors of survival (women/Miss/Mrs survived at much higher rates)
- **Pclass** strongly correlates with survival and fare
- **Small families (2–4)** had better survival rates than solo travellers or large families
- **Children** had elevated survival rates compared to adults
- **Fare (log-transformed)** has a clear positive relationship with survival probability
